# LHCb $D_s^+\to\pi^-\pi^+\pi^+$ amplitude analysis model

Paper: **LHCb, JHEP 07 (2023) 204, arXiv:2209.09840**.

This notebook is designed for a real-data fit. Edit the configuration cell to provide the ROOT data file/tree, the three Dalitz-invariant branches, the ROOT TH2 efficiency map, the ROOT TH2 background map, and the signal fraction from the $m(\pi\pi\pi)$ fit.

The nominal amplitude contains a 50-point QMI $\pi^+\pi^-$ S wave and $\rho(770)^0$, $\omega(782)$, $\rho(1450)^0$, $\rho(1700)^0$, $f_2(1270)$ and $f'_2(1525)$. The $\rho(770)$ uses Gounaris--Sakurai; the other explicit resonances use RBW. The Blatt--Weisskopf radii are $r_D=5.0$ and $r_R=1.5$ GeV$^{-1}$. The identical $\pi^+$ pair is symmetrized automatically by DalitzPlotFitter.

**Fidelity notes.** The publication smooths the 15x15 efficiency/background surfaces with a 2D cubic spline, while the current ROOT TH2 helpers evaluate the supplied map bin-by-bin; use the final smoothed maps if you have them. The paper also includes a 2.3 MeV mass-resolution convolution for the narrow $\omega(782)$; the current coherent-amplitude API does not yet apply that amplitude-level convolution, so this notebook uses the nominal $\omega$ RBW without it.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from dalitzplotfitter import (
    BackgroundSpec, DecayChannel, DecayModel, FitSession, GounarisSakurai,
    Parameter, QMI, RealImag, Resonance, enable_x64,
    histogram_background_from_root, histogram_efficiency_from_root,
    read_phase_space_sample, read_root_histogram2d,
)

enable_x64()
print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())


## 1. Input configuration
Edit this cell for your ROOT inputs. The Dalitz invariants and histogram axes are assumed to be in GeV$^2$.


In [ ]:
DATA_FILE = Path('/path/to/Ds2pipipi_data.root')
DATA_TREE = 'DecayTree'
S12_BRANCH, S13_BRANCH, S23_BRANCH = 's12', 's13', 's23'
DATA_CUT = None
ENTRY_START, ENTRY_STOP = None, None

EFFICIENCY_FILE = Path('/path/to/efficiency.root')
EFFICIENCY_HIST = 'efficiency'
BACKGROUND_FILE = Path('/path/to/background.root')
BACKGROUND_HIST = 'background'

# Replace by the value obtained from your three-pion mass fit.
SIGNAL_FRACTION = 0.95

# Conventional mass-Dalitz Gauss--Legendre normalization for this D decay.
NORMALIZATION_BIN_WIDTH = 0.005
RUN_FIT = True
USE_SIMPLEX = False
NCALL = 250_000
TOLERANCE = 1e-4
PROJECTION_SIZE = 100_000


## 2. Published QMI S-wave starting solution
The 50 mass points, magnitudes and phases below are the central values from Table 9. The displayed phases are unwrapped before interpolation so a $+180^\circ/-180^\circ$ display wrap is not interpreted as a physical discontinuity. `QMI(..., interpolation='linear')` interpolates magnitude and phase independently in $s=m^2$.


In [ ]:
QMI_KNOTS_GEV = np.array([0.280,0.390,0.470,0.546,0.623,0.698,0.766,0.819,0.865,0.900,0.925,0.942,0.955,0.964,0.972,0.978,0.983,0.990,1.001,1.023,1.051,1.083,1.116,1.149,1.179,1.205,1.227,1.245,1.261,1.276,1.289,1.302,1.314,1.326,1.338,1.351,1.363,1.375,1.387,1.399,1.411,1.424,1.437,1.450,1.465,1.484,1.511,1.554,1.613,1.823])
QMI_MAGNITUDE_PAPER = np.array([4.54,4.05,4.10,4.41,4.69,4.691,4.994,5.43,6.405,8.096,10.624,13.47,16.56,19.45,22.15,24.62,26.95,20.89,13.695,10.995,9.593,8.731,7.606,6.961,6.515,6.506,6.264,6.125,6.081,6.071,5.912,5.893,5.901,6.031,5.904,6.086,6.181,6.185,6.60,6.63,6.90,7.14,7.22,7.56,7.33,7.13,5.009,2.456,2.31,3.75])
QMI_PHASE_DEG_PAPER = np.array([176.8,152.8,147.6,146.4,149.6,157.4,169.5,-172.8,-152.0,-133.0,-116.5,-103.0,-88.8,-74.9,-59.5,-56.7,-21.8,-9.8,5.45,11.28,21.57,36.27,48.02,54.42,63.46,63.47,72.45,72.80,77.2,82.2,86.1,88.8,93.2,93.8,98.7,103.3,105.7,110.4,114.5,119.7,126.5,132.3,142.03,153.74,166.50,-172.15,-136.8,-126.8,-99.5,3.0])
QMI_PHASE_RAD_START = np.unwrap(np.deg2rad(QMI_PHASE_DEG_PAPER))
assert len(QMI_KNOTS_GEV) == len(QMI_MAGNITUDE_PAPER) == len(QMI_PHASE_RAD_START) == 50

fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax[0].plot(QMI_KNOTS_GEV, QMI_MAGNITUDE_PAPER, 'o-')
ax[0].set(xlabel=r'$m(\pi^+\pi^-)$ [GeV]', ylabel='S-wave magnitude')
ax[1].plot(QMI_KNOTS_GEV, np.rad2deg(QMI_PHASE_RAD_START), 'o-')
ax[1].set(xlabel=r'$m(\pi^+\pi^-)$ [GeV]', ylabel='unwrapped phase [deg]')
plt.show()


## 3. Nominal amplitude model
The paper uses $f_2(1270)\pi^+$ as the fixed reference amplitude ($|c|=1$, phase $0$). `normalize_components=False` is intentional: it preserves the paper's raw coefficient convention instead of rescaling each component to unit integral. The 100 QMI knot parameters plus five non-reference complex coefficients give 110 free parameters.


In [ ]:
def polar_coefficient(name, magnitude, phase_deg, *, fixed=False):
    phase = np.deg2rad(phase_deg)
    x, y = float(magnitude*np.cos(phase)), float(magnitude*np.sin(phase))
    if fixed:
        return RealImag(x, y)
    return RealImag(
        Parameter.coefficient(f'{name}.x', x, owner=name, step=0.01),
        Parameter.coefficient(f'{name}.y', y, owner=name, step=0.01),
    )

S_OWNER = 'pipi_S_qmi'
s_magnitudes = tuple(
    Parameter.dynamics(f'S.mag_{i:02d}', float(v), owner=S_OWNER, bounds=(0.0, None), step=max(0.02, 0.01*float(v)))
    for i, v in enumerate(QMI_MAGNITUDE_PAPER)
)
s_phases = tuple(
    Parameter.dynamics(f'S.phase_{i:02d}', float(v), owner=S_OWNER, step=np.deg2rad(1.0))
    for i, v in enumerate(QMI_PHASE_RAD_START)
)
s_wave = QMI(tuple(QMI_KNOTS_GEV), s_magnitudes, s_phases, interpolation='linear')

coeff = {
    'rho770': polar_coefficient('rho770', 0.1201, 79.4),
    'omega782': polar_coefficient('omega782', 0.04001, -109.9),
    'rho1450': polar_coefficient('rho1450', 1.277, -115.2),
    'rho1700': polar_coefficient('rho1700', 0.873, -60.9),
    'f2_1270': polar_coefficient('f2_1270', 1.0, 0.0, fixed=True),
    'f2p_1525': polar_coefficient('f2p_1525', 0.1098, 178.1),
}

channel = DecayChannel('D_s+', ('pi-', 'pi+', 'pi+'))
R_D, R_R = 5.0, 1.5
model = DecayModel(
    channel,
    [
        Resonance(S_OWNER, (0,1), RealImag(1.0,0.0), mass=1.0, width=0.0, spin=0, lineshape=s_wave, resonance_radius=R_R, parent_radius=R_D),
        Resonance('rho770', (0,1), coeff['rho770'], mass=0.77526, width=0.1491, spin=1, lineshape=GounarisSakurai(), resonance_radius=R_R, parent_radius=R_D),
        Resonance('omega782', (0,1), coeff['omega782'], mass=0.78265, width=0.00849, spin=1, resonance_radius=R_R, parent_radius=R_D),
        Resonance('rho1450', (0,1), coeff['rho1450'], mass=1.465, width=0.400, spin=1, resonance_radius=R_R, parent_radius=R_D),
        Resonance('rho1700', (0,1), coeff['rho1700'], mass=1.720, width=0.250, spin=1, resonance_radius=R_R, parent_radius=R_D),
        Resonance('f2_1270', (0,1), coeff['f2_1270'], mass=1.2755, width=0.1867, spin=2, resonance_radius=R_R, parent_radius=R_D),
        Resonance('f2p_1525', (0,1), coeff['f2p_1525'], mass=1.5174, width=0.086, spin=2, resonance_radius=R_R, parent_radius=R_D),
    ],
    normalize_components=False,
    normalization_method='gauss-legendre',
    normalization_bin_width=NORMALIZATION_BIN_WIDTH,
)
print('components:', [c.name for c in model.components])
print('free model parameters:', sum(not p.fixed for p in model.parameters))
assert sum(not p.fixed for p in model.parameters) == 110


## 4. Load data and the ROOT TH2 maps
`FitSession` includes the efficiency in the signal normalization and normalizes the background surface automatically.


In [ ]:
for path, label in [(DATA_FILE,'data'),(EFFICIENCY_FILE,'efficiency'),(BACKGROUND_FILE,'background')]:
    if not path.exists():
        raise FileNotFoundError(f'{label} file does not exist: {path}. Edit the configuration cell.')

data = read_phase_space_sample(
    DATA_FILE, DATA_TREE, s12=S12_BRANCH, s13=S13_BRANCH, s23=S23_BRANCH,
    cut=DATA_CUT, entry_start=ENTRY_START, entry_stop=ENTRY_STOP,
)
efficiency = histogram_efficiency_from_root(EFFICIENCY_FILE, EFFICIENCY_HIST, x_variable='s12', y_variable='s13')
background = histogram_background_from_root(BACKGROUND_FILE, BACKGROUND_HIST, x_variable='s12', y_variable='s13')

print(f'Loaded {data.size:,} candidates')
for name in ('s12','s13','s23'):
    a = np.asarray(getattr(data, name))
    print(f'{name}: [{a.min():.6g}, {a.max():.6g}] GeV^2')
if max(float(jnp.max(data.s12)), float(jnp.max(data.s13)), float(jnp.max(data.s23))) > 100.0:
    raise ValueError('Dalitz invariants look too large for GeV^2; convert MeV^2 branches before fitting.')

eff_v, eff_x, eff_y = read_root_histogram2d(EFFICIENCY_FILE, EFFICIENCY_HIST)
bkg_v, bkg_x, bkg_y = read_root_histogram2d(BACKGROUND_FILE, BACKGROUND_HIST)
fig, ax = plt.subplots(1,3,figsize=(16,4.5),constrained_layout=True)
ax[0].hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=100)
ax[0].set(xlabel=r'$s_{12}$ [GeV$^2$]', ylabel=r'$s_{13}$ [GeV$^2$]', title='data')
im=ax[1].pcolormesh(np.asarray(eff_x),np.asarray(eff_y),np.asarray(eff_v).T,shading='auto'); ax[1].set_title('efficiency'); fig.colorbar(im,ax=ax[1])
im=ax[2].pcolormesh(np.asarray(bkg_x),np.asarray(bkg_y),np.asarray(bkg_v).T,shading='auto'); ax[2].set_title('background'); fig.colorbar(im,ax=ax[2])
plt.show()


## 5. Likelihood and fit
The signal fraction is fixed, as in the analysis strategy: $P=f_{\rm sig}P_{\rm sig}+(1-f_{\rm sig})P_{\rm bkg}$ with $P_{\rm sig}\propto |A|^2\epsilon$. The fit starts from the published central solution.


In [ ]:
session = FitSession(
    model, data, efficiency=efficiency, signal_fraction=SIGNAL_FRACTION,
    backgrounds=(BackgroundSpec('sideband', background),),
)
_ = session.objective  # prepare host-side caches before the Minimizer JIT
start = {p.name: float(p.value) for p in session.parameters if not p.fixed}
print('free fit parameters:', len(start))

if RUN_FIT:
    result = session.fit(start, simplex=USE_SIMPLEX, ncall=NCALL, tolerance=TOLERANCE, verbose=1)
    report = session.report(result, acceptance_weighted_fractions=False, include_correlation=False)
else:
    result, report = None, None
    print('RUN_FIT=False: fit skipped')


## 6. Compare fitted explicit-resonance coefficients with Table 7


In [ ]:
PAPER_COEFF = {'rho770':(0.1201,79.4),'omega782':(0.04001,-109.9),'rho1450':(1.277,-115.2),'rho1700':(0.873,-60.9),'f2_1270':(1.0,0.0),'f2p_1525':(0.1098,178.1)}
if result is not None:
    values = session.result_values(result)
    print(f"{'component':14s} {'fit |c|':>12s} {'fit phase':>12s} {'paper |c|':>12s} {'paper phase':>12s}")
    for name,(pmag,pphase) in PAPER_COEFF.items():
        x,y = (1.0,0.0) if name=='f2_1270' else (values[f'{name}.x'],values[f'{name}.y'])
        mag=float(np.hypot(x,y)); phase=(float(np.rad2deg(np.arctan2(y,x)))+180)%360-180
        print(f'{name:14s} {mag:12.5g} {phase:12.3f} {pmag:12.5g} {pphase:12.3f}')


## 7. QMI S wave, Argand diagram, fit fractions and projections


In [ ]:
if result is not None:
    values = session.result_values(result)
    fit_mag=np.array([values[f'S.mag_{i:02d}'] for i in range(50)])
    fit_phase=np.array([values[f'S.phase_{i:02d}'] for i in range(50)])
    fit_complex=fit_mag*np.exp(1j*fit_phase)
    paper_complex=QMI_MAGNITUDE_PAPER*np.exp(1j*QMI_PHASE_RAD_START)
    fig,ax=plt.subplots(1,3,figsize=(16,4.5),constrained_layout=True)
    ax[0].plot(QMI_KNOTS_GEV,QMI_MAGNITUDE_PAPER,'o',label='paper'); ax[0].plot(QMI_KNOTS_GEV,fit_mag,'-',label='fit'); ax[0].legend(); ax[0].set_ylabel('magnitude')
    ax[1].plot(QMI_KNOTS_GEV,np.rad2deg(QMI_PHASE_RAD_START),'o',label='paper'); ax[1].plot(QMI_KNOTS_GEV,np.rad2deg(fit_phase),'-',label='fit'); ax[1].legend(); ax[1].set_ylabel('phase [deg]')
    ax[2].plot(paper_complex.real,paper_complex.imag,'o',label='paper'); ax[2].plot(fit_complex.real,fit_complex.imag,'-',label='fit'); ax[2].set(xlabel=r'Re $A_S$',ylabel=r'Im $A_S$'); ax[2].legend()
    for a in ax[:2]: a.set_xlabel(r'$m(\pi^+\pi^-)$ [GeV]')
    plt.show()

    session.print_fit_fractions(result, acceptance_weighted=False, include_interference=True, precision=4)
    for variable in ('s12','s13','s23'):
        session.plot_projection(result, variable, bins=60, projection_size=PROJECTION_SIZE)
        plt.show()


## 8. Publication checks
Nominal statistical fit fractions from Table 3 are approximately: S wave 84.97%, $\rho(770)^0$ 1.038%, $\omega(782)$ 0.360%, $\rho(1450)^0$ 3.86%, $\rho(1700)^0$ 0.365%, $f_2(1270)$ 13.69%, and $f'_2(1525)$ 0.0455%; the sum is 104.3% because of interference. Reproducing those values requires the same selected sample, exact signal fraction, efficiency/background surfaces, and the publication's $\omega$ resolution treatment.
